# 02 - Preprocessing & Feature Engineering

Cleans the raw data and builds every feature used downstream by the recommenders. Uses `src/data_preprocessing.py` and `src/feature_engineering.py` so the exact same code path is used here, in `run_pipeline.py`, and in the Streamlit app.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import data_preprocessing, feature_engineering
from utils import PROCESSED_DIR

## 1. Clean raw data

Removes duplicates, fills missing brand/description, drops invalid prices, clips unrealistic ratings (outside 1-5), and removes interactions referencing unknown users/products.

In [ ]:
products, users, interactions = data_preprocessing.run()
print('Cleaned shapes:', products.shape, users.shape, interactions.shape)

## 2. Train/test split (leakage-safe)

A **time-based, per-user** split: each user's most recent 20% of interactions become the test set, the rest is train. This means the model is always evaluated on interactions that happened AFTER everything it was trained on for that user — never leaking future behavior into training.

In [ ]:
train, test = feature_engineering.build_train_test_split(interactions)
print(f'Train: {len(train)} rows | Test: {len(test)} rows')
print(f'Test set covers {test.user_id.nunique()} of {interactions.user_id.nunique()} users')

## 3. Feature engineering

Builds product-level features (avg_rating, weighted popularity, popularity_rank, price_bucket, combined_text for TF-IDF) and user-level features (activity_score, top_category, avg_rating_given), computed **only from the training split** to avoid leakage.

In [ ]:
product_features = feature_engineering.build_product_features(products, train)
user_features = feature_engineering.build_user_features(users, train, products)
category_pref = feature_engineering.build_category_preference_matrix(train, products)
product_features.head()

In [ ]:
user_features.head()

### Feature explanations

| Feature | What it means | Why it matters |
|---|---|---|
| `avg_rating` / `n_ratings` | Mean star rating and count of ratings for a product | Quality signal, used in cold-start & explanations |
| `weighted_interaction_score` | Sum of interaction weights (purchase=5, wishlist=3, cart=2, view=1) | Popularity signal, robust to noisy raw view counts |
| `popularity_rank` | Percentile rank of the above | Used to bucket 'trending' vs 'niche' products |
| `price_bucket` | Quartile-based Low/Medium/High/Premium | Lets the dashboard segment by price tier |
| `combined_text` | Concatenated title+brand+category+tags+description | Input to the TF-IDF content-based model |
| `activity_score` | Weighted sum of a user's interactions | Drives the cold-start confidence blending in the hybrid model |
| `top_category` | User's most-interacted-with category | Used for the 'because you frequently buy from X' explanation |


In [ ]:
product_features.to_csv(PROCESSED_DIR / 'products_features.csv', index=False)
user_features.to_csv(PROCESSED_DIR / 'users_features.csv', index=False)
train.to_csv(PROCESSED_DIR / 'interactions_train.csv', index=False)
test.to_csv(PROCESSED_DIR / 'interactions_test.csv', index=False)
print('Saved processed features.')